# 03 — Audit real exports and document the analysis protocol

Run after the pilot, then again after the full export completes. A successful structural audit does not prove physical accuracy, independent validation or RFI cause. The study target is a QA-passing SMAP product observation at a fixed target-grid cell/date, not a regional mean copied to multiple pixels.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util, json
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent, Path("/content/cht-soil-moisture-gapfilling")]
             if (p / "pyproject.toml").exists()), None)
if ROOT is None and IN_COLAB:
    ROOT = Path("/content/cht-soil-moisture-gapfilling")
    subprocess.run(["git", "clone", "--branch", "research-notebooks-v2", "--single-branch",
                    "https://github.com/Bipul-roy-AI/cht-soil-moisture-gapfilling.git", str(ROOT)], check=True)
if ROOT is None:
    raise RuntimeError("Open this notebook from the cloned repository, or run it in Colab.")
ROOT = ROOT.resolve()
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[gee,notebooks]"], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Project:", ROOT)


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/CHT_SM_v2")
else:
    WORK = ROOT / "work_v2"
WORK.mkdir(parents=True, exist_ok=True)
print("Persistent work folder:", WORK)


In [ ]:
from chtsm.data import read_config, load_exports, audit_frame, save_json
config = read_config(WORK / "study_config.json")
# PILOT CHECK: set the actual downloaded pilot CSV path, or leave as None.
PILOT_CSV = None
if PILOT_CSV is not None:
    pilot = load_exports([Path(PILOT_CSV)], config)
    display(pilot[["date","cell_id","sm_raw","sm_qa","target_sm","missing_reason",
                   "raw_vv_db","rtc_vv_db","evi","sar_valid_fraction"]].head(30))
    print(pilot.missing_reason.value_counts())
    print("Pilot rows:", len(pilot), "| no full-calendar assertion at this stage.")

In [ ]:
from chtsm.data import read_config, load_exports, audit_frame, save_json
CONFIG_PATH = WORK / "study_config.json"
config = read_config(CONFIG_PATH)
# Move the monthly FULL exports from the Drive export folder into this folder.
# Pilot exports have a different prefix and must not be mixed into the full table.
EXPORT_DIR = WORK / "exports"
CSV_PATHS = sorted(EXPORT_DIR.glob("chtsm_v2_full_*.csv"))
if not CSV_PATHS:
    raise FileNotFoundError("Put completed monthly chtsm_v2_full_*.csv exports in WORK/exports first.")
frame = load_exports(CSV_PATHS, config)


In [ ]:
report = audit_frame(frame, config, require_complete=True)
save_json(WORK / "full_data_audit.json", report)
print(json.dumps(report, indent=2, default=str))
display(frame.groupby("split").agg(rows=("cell_id","count"), dates=("date","nunique"),
                                    usable_targets=("target_sm","count")))
# Counts vary by date: a mean over available pixels can change support.
availability = frame.groupby("date").target_sm.agg(["count", "size"])
availability["usable_fraction"] = availability["count"] / availability["size"]
availability.to_csv(WORK / "observed_support_by_date.csv")
import matplotlib.pyplot as plt
availability.usable_fraction.plot(figsize=(12,3), ylabel="Fraction of selected cells with usable SMAP", title="Availability audit — no cause attribution")
plt.show()

In [ ]:
# Inspect persistent missingness and sensor support BEFORE modelling.
cell_summary = frame.groupby("cell_id").agg(lon=("lon","first"),lat=("lat","first"),
    usable_fraction=("target_sm",lambda s:s.notna().mean()),forest_fraction=("forest_fraction","first"),
    slope_deg=("slope_deg","first"),mean_radar_coverage=("sar_valid_fraction","mean"))
cell_summary.to_csv(WORK / "cell_support_audit.csv")
display(cell_summary.sort_values("usable_fraction").head(20))
print("Revisit/coverage, QA rejection, and native diagnostic/RFI reasons need separate interpretation.")

Write the protocol before training or final-test inspection: target definition, primary macro-cell RMSE, planned algorithms/feature groups/seeds, gap lengths, comparison information sets, subgroup definitions and uncertainty method. The training notebook uses validation only for model selection, then a separate calibration year. Final evaluation requires a local freeze; this is not an external preregistration. If spatial transfer is planned, choose held-out cells based on geometry and predeclare it before opening final-test results.

In [ ]:
PROTOCOL_NOTE = (
    "Primary task: retrospective reconstruction of QA-passing SMAP cell/date targets. "
    "Primary selection outcome: macro-cell RMSE on validation year only. "
    "Compare climate-only, climate+raw SAR and climate+RTC SAR across all declared seeds, "
    "with mean/seasonal/ERA5 baselines. Separate calibration year; final test year untouched. "
    "Artificial 1/3/7/14-day gaps are secondary analyses; long gaps are stress tests. "
    "No in-situ accuracy or physical-causation claim without independent evidence."
)
(WORK / "protocol_note.txt").write_text(PROTOCOL_NOTE + "\n")
print(PROTOCOL_NOTE)